# Notebook 11 — `11_instance_mixed_effect_growth`

## Overview

**Purpose:** Robustness check on the RQ2 growth result. The two-way fixed-effects
growth model is re-estimated as a linear mixed model (random regency intercept,
year fixed effects) to confirm that the flood conclusion does not depend on how
regency heterogeneity is modelled. This is an *inferential* robustness check
(coefficients, SE, ICC, AIC/BIC), not a predictive model, and it is compared
against the fixed-effects model, not against XGBoost. Consistent with the
manuscript, the growth flood block stays jointly significant (driven by physical
damage) under both estimators, while the within-R² stays near zero.

## Data requirements

This notebook runs on the model-ready panel exported by Notebook 8; no raw data
download is required here.

| Input | Produced by | Directory |
|---|---|---|
| Model-ready panel (`transformed` sheet) | Notebook 8 | `data/processed/8_panel_data.xlsx` |

## Workflow

| Step | Description |
|---|---|
| 1 | Load the panel and fit the two-way fixed-effects benchmark (flood-block Wald) |
| 2 | Fit the linear mixed model (random regency intercept, year fixed, ML) |
| 3 | FE vs mixed coefficient comparison (flood variables first) |
| 4 | Coefficient-stability diagnostic (not a Hausman test) |
| 5 | Between- vs within-regency variation (explains any control movement) |
| 6 | What to report |

## Outputs

No files are written; the notebook reports coefficients, the flood-block joint
test, ICC and fit statistics for the manuscript's robustness paragraph.

In [2]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

import statsmodels.formula.api as smf
from scipy import stats
from linearmodels.panel import PanelOLS

RANDOM_SEED = 42

panel = pd.read_excel("../data/processed/8_panel_data.xlsx", sheet_name="transformed").sort_values(
    ["bps_kab_code", "year"]).reset_index(drop=True)

FLOOD    = ["event_count_log", "HCI_log", "PDI_log"]
CONTROLS = ["log_population", "rls_years"]
X        = FLOOD + CONTROLS
Y        = "grdp_growth_pct"

print(f"Panel: {len(panel):,} rows, {panel.bps_kab_code.nunique()} regencies, "
      f"{panel.year.nunique()} years")

Panel: 5,140 rows, 514 regencies, 10 years


## STEP 1 — Fixed-effects main model (benchmark)

Two-way FE (regency + year), clustered SE by regency. The mixed model is checked
against this. **Reading the output:** `beta` is the within-regency association
net of regency and year effects; `p<0.05` flags significance; the small
within-R² (~0.009) is part of the finding, not a defect. The **flood-block joint
Wald test** (H0: all three flood coefficients = 0) is the test for the flood
hypothesis as a whole — read it rather than any single control's star.

In [4]:
# ═════════════════════════════════════════════════════════════
# STEP 1 — Fixed-effects main model (benchmark)
# ═════════════════════════════════════════════════════════════
pdat = panel.set_index(["bps_kab_code", "year"])
sub  = pdat[[Y] + X].dropna()

fe = PanelOLS(sub[Y], sub[X], entity_effects=True, time_effects=True,
              drop_absorbed=True).fit(cov_type="clustered", cluster_entity=True)

print("FIXED EFFECTS (two-way, clustered SE)")
print("=" * 52)
print(f"{'variable':<18}{'beta':>9}{'p':>9}   sig  type")
print("-" * 47)
for c in X:
    sig = "***" if fe.pvalues[c]<0.01 else "**" if fe.pvalues[c]<0.05 else "*" if fe.pvalues[c]<0.10 else ""
    print(f"{c:<18}{fe.params[c]:>+9.4f}{fe.pvalues[c]:>9.4f}   {sig:<4} [{'flood' if c in FLOOD else 'control'}]")
print(f"\n  within-R2 = {fe.rsquared_within:.4f}    (*** p<.01, ** p<.05, * p<.10)")

# --- flood-block joint Wald (H0: all flood coefs = 0; controls held fixed) ---
w_fe = fe.wald_test(formula=" = ".join(FLOOD) + " = 0")
p_fe_blk = float(w_fe.pval)
print(f"\n  [FLOOD-BLOCK joint] chi2({len(FLOOD)}) = {float(w_fe.stat):.3f}  "
      f"p = {p_fe_blk:.4g}  {'sig' if p_fe_blk<0.05 else 'n.s.'}")

FIXED EFFECTS (two-way, clustered SE)
variable               beta        p   sig  type
-----------------------------------------------
event_count_log     +0.0711   0.2135        [flood]
HCI_log             -0.0324   0.1203        [flood]
PDI_log             -0.0216   0.0192   **   [flood]
log_population      -0.5785   0.3843        [control]
rls_years           -0.1777   0.3510        [control]

  within-R2 = 0.0092    (*** p<.01, ** p<.05, * p<.10)

  [FLOOD-BLOCK joint] chi2(3) = 13.646  p = 0.003429  sig


## STEP 2 — Linear mixed-effects model

Random regency intercept via `groups`; **year as fixed effects via `C(year)`**,
matching the two-way FE benchmark. Year effects capture common temporal shocks
(COVID, inflation, El Niño) that need not be linear — so `C(year)` is used, not a
linear year term.

Predictors are standardised (`_z`) **only to help the optimiser converge**;
coefficients are converted back to original units before reporting, so these are
NOT standardised betas. The model is fit by **ML (`reml=False`)** so that AIC/BIC
are well defined (REML log-likelihoods are not comparable across models with
different fixed-effect structures).

In [6]:
# ═════════════════════════════════════════════════════════════
# STEP 2 — Linear mixed-effects model
# ═════════════════════════════════════════════════════════════
df = sub.reset_index().copy()
for c in X:
    df[c + "_z"] = (df[c] - df[c].mean()) / df[c].std()   # for optimisation only
XZ = [c + "_z" for c in X]

# Year as fixed effects (C(year)), matching the FE benchmark's time_effects.
# ML (reml=False) so AIC/BIC are valid. With a full year-dummy set the design
# matrix is near-singular, so powell (derivative-free) is tried first; lbfgs is
# a fallback for the cases where it converges faster.
me = None
for _method in ["powell", "lbfgs", "nm"]:
    try:
        _fit = smf.mixedlm(f"{Y} ~ {' + '.join(XZ)} + C(year)",
                           data=df, groups=df["bps_kab_code"]).fit(
                           method=_method, reml=False)
        if _fit.converged:
            me = _fit
            print(f"Converged with method='{_method}', year as C(year), ML.")
            break
    except Exception as e:
        print(f"  method='{_method}' not usable ({str(e)[:40]}), trying next solver...")
assert me is not None, "mixed model did not converge with any method"

var_re = float(me.cov_re.iloc[0, 0]); var_e = float(me.scale)
icc = var_re / (var_re + var_e)

print("\nMIXED EFFECTS (random regency intercept, year fixed)")
print("=" * 52)
print(f"{'variable':<18}{'beta':>9}{'p':>9}   sig  type")
print("-" * 47)
for c in X:
    beta          = me.params[c + "_z"]          # standardised
    beta_original = beta / df[c].std()           # back to original units
    p = me.pvalues[c + "_z"]
    sig = "***" if p<0.01 else "**" if p<0.05 else "*" if p<0.10 else ""
    print(f"{c:<18}{beta_original:>+9.4f}{p:>9.4f}   {sig:<4} [{'flood' if c in FLOOD else 'control'}]")
print(f"\n  ICC = {icc:.3f}   (share of variance between regencies)")
print(f"  AIC = {me.aic:.1f}   BIC = {me.bic:.1f}   logLik = {me.llf:.1f}")

# --- flood-block joint Wald on the mixed fit (H0 all=0 is scale-invariant) ---
fz   = [c + "_z" for c in FLOOD]
b    = me.params[fz].values
cov  = me.cov_params().loc[fz, fz].values
wmix = float(b @ np.linalg.inv(cov) @ b)
p_mix_blk = float(stats.chi2.sf(wmix, len(fz)))
print(f"\n  [FLOOD-BLOCK joint] chi2({len(FLOOD)}) = {wmix:.3f}  "
      f"p = {p_mix_blk:.4g}  {'sig' if p_mix_blk<0.05 else 'n.s.'}")

# --- headline agreement line ---
agree = (p_fe_blk < 0.05) == (p_mix_blk < 0.05)
verdict = ("ROBUST (jointly sig under BOTH)" if (p_fe_blk<0.05 and p_mix_blk<0.05)
           else "NULL (n.s. under BOTH)" if (p_fe_blk>=0.05 and p_mix_blk>=0.05)
           else "MIXED (sig under one only)")
print(f"\n  FLOOD-BLOCK  FE p={p_fe_blk:.4g}  |  Mixed p={p_mix_blk:.4g}  "
      f"-> conclusion {'AGREES' if agree else 'DIFFERS'}: {verdict}")

Converged with method='powell', year as C(year), ML.

MIXED EFFECTS (random regency intercept, year fixed)
variable               beta        p   sig  type
-----------------------------------------------
event_count_log     +0.0834   0.1070        [flood]
HCI_log             -0.0341   0.1631        [flood]
PDI_log             -0.0217   0.0129   **   [flood]
log_population      +0.0700   0.1511        [control]
rls_years           +0.0781   0.0080   ***  [control]

  ICC = 0.308   (share of variance between regencies)
  AIC = 19645.3   BIC = 19756.6   logLik = -9805.7

  [FLOOD-BLOCK joint] chi2(3) = 12.782  p = 0.005132  sig

  FLOOD-BLOCK  FE p=0.003429  |  Mixed p=0.005132  -> conclusion AGREES: ROBUST (jointly sig under BOTH)


Converged with method='powell', year as C(year), ML.

MIXED EFFECTS (random regency intercept, year fixed)
variable               beta        p   sig  type
-----------------------------------------------
event_count_log     +0.0834   0.1070        [flood]
HCI_log             -0.0341   0.1631        [flood]
PDI_log             -0.0217   0.0129   **   [flood]
log_population      +0.0700   0.1511        [control]
rls_years           +0.0781   0.0080   ***  [control]

  ICC = 0.308   (share of variance between regencies)
  AIC = 19645.3   BIC = 19756.6   logLik = -9805.7

  [FLOOD-BLOCK joint] chi2(3) = 12.782  p = 0.005132  sig

  FLOOD-BLOCK  FE p=0.003429  |  Mixed p=0.005132  -> conclusion AGREES: ROBUST (jointly sig under BOTH)


## STEP 3 — FE vs Mixed — coefficient comparison

Focus on the **flood** variables (the RQ). The `abs_diff` and status are computed
automatically, so the label always matches the numbers — nothing is hard-coded.

In [8]:
# ═════════════════════════════════════════════════════════════
# STEP 3 — FE vs Mixed — coefficient comparison
# ═════════════════════════════════════════════════════════════
rows = []
for c in X:
    b_fe, p_fe = float(fe.params[c]), float(fe.pvalues[c])
    b_me = float(me.params[c + "_z"] / df[c].std())
    p_me = float(me.pvalues[c + "_z"])
    abs_diff = abs(b_fe - b_me)
    if   abs_diff < 0.05: status = "very similar"
    elif abs_diff < 0.20: status = "moderately different"
    else:                 status = "substantially different"
    note = []
    if (b_fe * b_me) < 0:              note.append("direction differs")
    if p_fe < 0.05 and p_me >= 0.05:   note.append("FE only")
    if p_me < 0.05 and p_fe >= 0.05:   note.append("Mixed only")
    rows.append({"variable": c, "type": "flood" if c in FLOOD else "control",
                 "FE_b": b_fe, "FE_p": p_fe, "ME_b": b_me, "ME_p": p_me,
                 "abs_diff": abs_diff, "status": status, "note": ", ".join(note) or "-"})
comp = pd.DataFrame(rows)
print(comp.to_string(index=False, formatters={
    "FE_b": "{:+.3f}".format, "FE_p": "{:.3f}".format,
    "ME_b": "{:+.3f}".format, "ME_p": "{:.3f}".format, "abs_diff": "{:.3f}".format}))

print("\nFlood variables:", ", ".join(f"{r.variable} ({r.status})" for _, r in
      comp[comp.type == "flood"].iterrows()))
print("Controls differ more — expected, and explained by the variance")
print("decomposition in Section 5.")

       variable    type   FE_b  FE_p   ME_b  ME_p abs_diff                  status                          note
event_count_log   flood +0.071 0.214 +0.083 0.107    0.012            very similar                             -
        HCI_log   flood -0.032 0.120 -0.034 0.163    0.002            very similar                             -
        PDI_log   flood -0.022 0.019 -0.022 0.013    0.000            very similar                             -
 log_population control -0.579 0.384 +0.070 0.151    0.648 substantially different             direction differs
      rls_years control -0.178 0.351 +0.078 0.008    0.256 substantially different direction differs, Mixed only

Flood variables: event_count_log (very similar), HCI_log (very similar), PDI_log (very similar)
Controls differ more — expected, and explained by the variance
decomposition in Section 5.


## STEP 4 — Coefficient-stability diagnostic (FE vs Mixed)

**Not a Hausman test.** The FE-vs-RE choice is made by the Mundlak (CRE) test in
the FE notebook. This is only a descriptive check of whether the flood
coefficients move when the estimation framework changes — a robustness /
sensitivity diagnostic that complements the flood-block joint test above.

The Hausman-type quadratic form requires `Var(FE) − Var(Mixed)` to be
positive-definite. When it is not, the statistic is not χ²-distributed and no
p-value is reported — stability is judged from the coefficient table instead.

In [10]:
# ═════════════════════════════════════════════════════════════
# STEP 4 — Coefficient-stability diagnostic (FE vs Mixed)
# ═════════════════════════════════════════════════════════════
def stability_diagnostic(terms):
    b_fe = np.array([fe.params[c] for c in terms])
    b_me = np.array([me.params[c + "_z"] / df[c].std() for c in terms])
    v_fe = np.diag([fe.std_errors[c] ** 2 for c in terms])
    v_me = np.diag([(me.bse[c + "_z"] / df[c].std()) ** 2 for c in terms])
    dvar = v_fe - v_me
    diff = b_fe - b_me
    eig = np.linalg.eigvalsh(dvar)
    pd_ok = np.all(eig > 0)
    stat = float(diff @ np.linalg.pinv(dvar) @ diff)
    return stat, pd_ok, diff

stat, pd_ok, diff = stability_diagnostic(FLOOD)
print("COEFFICIENT-STABILITY DIAGNOSTIC (FE vs Mixed, flood only)")
print("=" * 60)
print("  max |FE - Mixed| across flood coefficients: "
      f"{np.max(np.abs(diff)):.4f}")
if pd_ok:
    p = 1 - stats.chi2.cdf(stat, len(FLOOD))
    print(f"  Hausman-type chi2({len(FLOOD)}) = {stat:.3f}, p = {p:.4f}")
    print("  -> flood coefficients " +
          ("differ" if p < 0.05 else "do not differ") + " significantly.")
else:
    print("  Var(FE)-Var(Mixed) is NOT positive-definite, so the Hausman-type")
    print("  quadratic form is not chi-square distributed. No p-value is reported.")
    print("  Judge stability descriptively from the coefficient table (Section 3):")
    print("  the flood coefficients are close in both magnitude and significance.")

COEFFICIENT-STABILITY DIAGNOSTIC (FE vs Mixed, flood only)
  max |FE - Mixed| across flood coefficients: 0.0124
  Var(FE)-Var(Mixed) is NOT positive-definite, so the Hausman-type
  quadratic form is not chi-square distributed. No p-value is reported.
  Judge stability descriptively from the coefficient table (Section 3):
  the flood coefficients are close in both magnitude and significance.


## STEP 5 — Why the controls differ but the flood coefficients don't

FE uses **within-regency** variation only. Mixed uses **within + between**. So a
regressor's two estimates diverge in proportion to how much of its variation is
*between* regencies.

In [12]:
# ═════════════════════════════════════════════════════════════
# STEP 5 — Why the controls differ but the flood coefficients don't
# ═════════════════════════════════════════════════════════════
print("BETWEEN- vs WITHIN-regency variation (SD)")
print("=" * 60)
print(f"{'variable':<18}{'between':>9}{'within':>9}{'ratio':>8}  reading")
print("-" * 58)
for c in X:
    within_sd  = df.groupby("bps_kab_code")[c].transform(lambda s: s - s.mean()).std()
    between_sd = df.groupby("bps_kab_code")[c].mean().std()
    ratio = between_sd / max(within_sd, 1e-9)
    if   ratio > 5:   reading = "mostly between variation"
    elif ratio < 1.5: reading = "substantial within variation"
    else:             reading = "mixed"
    print(f"{c:<18}{between_sd:>9.3f}{within_sd:>9.3f}{ratio:>7.1f}x  {reading}")

print()
print("Controls are dominated by between-regency variation (population barely")
print("changes within a regency over 10 years). FE removes that between variation,")
print("so the control coefficients are estimated from thin within variation and")
print("shift substantially. Flood variables carry substantial within variation, so")
print("FE and Mixed use largely the same signal -> the flood coefficients stay close.")

BETWEEN- vs WITHIN-regency variation (SD)
variable            between   within   ratio  reading
----------------------------------------------------------
event_count_log       0.570    0.580    1.0x  substantial within variation
HCI_log               0.386    0.924    0.4x  substantial within variation
PDI_log               3.091    3.637    0.8x  substantial within variation
log_population        1.022    0.073   14.0x  mostly between variation
rls_years             1.630    0.347    4.7x  mixed

Controls are dominated by between-regency variation (population barely
changes within a regency over 10 years). FE removes that between variation,
so the control coefficients are estimated from thin within variation and
shift substantially. Flood variables carry substantial within variation, so
FE and Mixed use largely the same signal -> the flood coefficients stay close.


event_count_log       0.570    0.580    1.0x  substantial within variation


HCI_log               0.386    0.924    0.4x  substantial within variation
PDI_log               3.091    3.637    0.8x  substantial within variation


log_population        1.022    0.073   14.0x  mostly between variation


rls_years             1.630    0.347    4.7x  mixed

Controls are dominated by between-regency variation (population barely
changes within a regency over 10 years). FE removes that between variation,
so the control coefficients are estimated from thin within variation and
shift substantially. Flood variables carry substantial within variation, so
FE and Mixed use largely the same signal -> the flood coefficients stay close.


## STEP 6 — What to report

**Framework.** The mixed-effects model uses all 5,140 regency-year observations
while accounting for the correlation of repeated observations within each regency
through a regency-specific random intercept. It is a robustness check on the
fixed-effects specification, evaluated by inferential quantities (coefficients,
SE, CI, ICC, AIC/BIC) — not by prediction metrics.

**Result (from this run).**

> Re-estimating the growth model as a linear mixed model with a random regency
> intercept and year fixed effects yields flood coefficients close to the
> fixed-effects estimates in both magnitude and significance: the flood block is
> **jointly significant under both estimators** (FE p≈0.003, mixed p≈0.005),
> driven by the Physical Damage Index, whose coefficient remains negative and
> significant in both models. The control coefficients differ more, consistent
> with their predominantly between-regency variation, which the mixed model uses
> and fixed effects removes. Because the Mundlak test shows regency effects are
> correlated with the regressors, fixed effects remains the main inferential
> specification; the mixed model confirms the flood conclusion — a robust but
> small association concentrated in physical damage — is not sensitive to this
> modelling choice. The within-R² remains near zero, so the association, though
> statistically robust, explains little of the year-on-year variation in growth.

**Contrast with the other outcomes (nb11 poverty/TPT).** For poverty and TPT the
same check returns a *non-significant* flood block under both estimators; for
growth it returns a *significant* one. The robustness framework is identical —
only the finding differs — which is exactly what supports the thesis that
statistical significance and predictive relevance are distinct.

**Terminology to keep straight:** the random intercept *accounts for* repeated
measurements — it is not a control variable. Do not call Section 4 a "Hausman
test"; it is a coefficient-stability diagnostic.